In [1]:
## importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

In [2]:
df = pd.read_excel('finalforms.xlsx')

In [3]:
# pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns',None)

In [4]:
print(df.shape)
# print(df.head())
df.columns[18]

(62, 104)


'Upon arrival, I prefer to interact with a human staff member rather than a digital system.'

In [5]:
# df

In [6]:
# Clean column names: strip spaces, replace line breaks, compress spaces
df.columns = (
    df.columns
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.strip()
    .str.replace(" +", " ", regex=True)
)

In [7]:
likert_cols = [
    "Upon arrival, I prefer to interact with a human staff member rather than a digital system",
    "It feels easier to know what to do when a person guides me upon arrival",
    "Personal interaction with staff at arrival makes the dining experience feel more genuine",
    "It is easier to understand menu items when a person explains them",
    "I like when a staff member helps me explore the menu or suggests dishes",
    "I trust human staff more than a digital system to provide accurate menu information",
    "I enjoy hearing about dishes from a person rather than reading them on a screen",
    "When a digital system fails, having human staff as a fallback makes me feel supported",
    "I like having the option of human assistance while using a digital system",
    "It is easier to explain my dietary preferences or special needs to a person",
    "It is easier to change or adjust my order when interacting with human staff",
    "I feel reassured when a person confirms my customized order verbally",
    "It is easier to get assistance during the meal from human staff than from a digital system",
    "Human staff are better at judging the right time to bring the next course than digital systems",
    "I feel more comfortable communicating personal or dietary needs to a person than through a digital device",
    "Being checked on by human staff during the meal makes the experience feel more personal",
    "It is easier to ask follow-up questions about dishes when speaking to a person",
    "I trust human staff to handle payments more accurately than a digital system",
    "It is easier to clarify billing or payment questions with a person",
    "If a payment issue occurs, I believe human staff can resolve it more easily than a digital system",
    "It is easier to understand the bill when a person explains what I am paying for",
    "Having a person handle payment makes me feel more confident and reassured about the transaction",
    "I feel more in control of my payment when interacting with a person rather than a digital system"
]


In [8]:
# Fix known naming mismatch FIRST
df = df.rename(columns={
    "Upon arrival, I prefer to interact with a human staff member rather than a digital system.":
    "Upon arrival, I prefer to interact with a human staff member rather than a digital system"
})

# Then check if column is actually present in the dataframe
present = [c for c in likert_cols if c in df.columns]
missing = [c for c in likert_cols if c not in df.columns]

print("Missing:", missing)
print("Found:", len(present), "of", len(likert_cols))

Missing: []
Found: 23 of 23


In [9]:
# missing = [c for c in likert_cols if c not in df.columns]
# extra   = [c for c in df.columns if c in likert_cols]

# missing, len(missing)


In [10]:
# target = "Upon arrival, I prefer to interact with a human staff member rather than a digital system"

# [c for c in df.columns if "Upon arrival" in c]


In [11]:
# df = df.rename(columns={
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system.":
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system"
# })


In [12]:
LIKERT_ORDER = {
    "strongly disagree": 1,
    "disagree": 2,
    "slightly disagree": 3,
    "slightly agree": 4,
    "agree": 5,
    "strongly agree": 6,
}


In [13]:
LIKERT_LABELS = {
    1: "strongly disagree",
    2: "disagree",
    3: "slightly disagree",
    4: "slightly agree",
    5: "agree",
    6: "strongly agree",
}


In [14]:
unique_responses = set()

for col in likert_cols:
    unique_responses.update(
        df[col]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )

sorted(unique_responses)




['agree',
 'disagree',
 'slightly agree',
 'slightly disagree',
 'strongly agree',
 'strongly disagree']

In [15]:
scale_keys = set(LIKERT_ORDER.keys())

unexpected = unique_responses - scale_keys
missing = scale_keys - unique_responses

unexpected, missing


(set(), set())

In [16]:
for col in likert_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )


In [17]:
for col in likert_cols:
    df[col] = df[col].map(LIKERT_ORDER)


In [18]:
#This computes how many missing values exist per Likert question column.

df[likert_cols].isna().sum()


Upon arrival, I prefer to interact with a human staff member rather than a digital system                    29
It feels easier to know what to do when a person guides me upon arrival                                      29
Personal interaction with staff at arrival makes the dining experience feel more genuine                     29
It is easier to understand menu items when a person explains them                                            29
I like when a staff member helps me explore the menu or suggests dishes                                      29
I trust human staff more than a digital system to provide accurate menu information                          30
I enjoy hearing about dishes from a person rather than reading them on a screen                              29
When a digital system fails, having human staff as a fallback makes me feel supported                        29
I like having the option of human assistance while using a digital system                               

In [19]:
# df = df.rename(columns={
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system.":
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system"
# })



p_vals = []

age_levels = df["Age band"].dropna().unique()

for q in likert_cols:
    groups = [
        df.loc[df["Age band"] == g, q].dropna()
        for g in age_levels
    ]
    # remove empty groups
    groups = [x for x in groups if len(x) > 0]

    if len(groups) >= 2:
        _, p = kruskal(*groups)
    else:
        p = np.nan

    p_vals.append(p)



In [20]:
# # groups

df["Age band"].unique()


p_vals

[0.9841868602151332,
 0.3672100543945057,
 0.44942925135423706,
 0.33474672508709263,
 0.5602583753826074,
 0.2728549461652404,
 0.9201442536349229,
 0.2214761522394015,
 0.7267154447969195,
 0.44729014473207596,
 0.30738158133793464,
 0.25424965203450745,
 0.5710202684116865,
 0.8200349854866675,
 0.9626835882599913,
 0.8715940421499073,
 0.735729792625291,
 0.06599720840899023,
 0.4837047807284802,
 0.6034295439017854,
 0.09958152579879757,
 0.05531405452019678,
 0.18328363999492445]

In [22]:
df_human_answered = df[df[likert_cols].notna().any(axis=1)]


In [23]:
# df_human_answered

In [24]:
age_counts = (
    df_human_answered["Age band"]
    .value_counts(dropna=False)
    .sort_index()
)

age_counts


Age band
18–24     1
25–34    14
35–44     7
45–54     6
55–64     3
65+       2
Name: count, dtype: int64

In [ ]:
# merging age bands

# Young:   18–34   → 1 + 14 = 15
# Middle:  35–54   → 7 + 6  = 13
# Older:   55+     → 3 + 2  = 5


In [25]:
def merge_age_band(age):
    if pd.isna(age):
        return np.nan
    elif age in ["18–24", "25–34"]:
        return "18–34"
    elif age in ["35–44", "45–54"]:
        return "35–54"
    elif age in ["55–64", "65+"]:
        return "55+"
    else:
        return np.nan

df_human_answered = df_human_answered.copy()

df_human_answered["Age_3grp"] = df_human_answered["Age band"].apply(merge_age_band)





In [33]:
# df_human_answered["Age_3grp"]

In [ ]:
# df_human_answered

In [ ]:
# df_human_answered["Age_3grp"]

In [34]:
# # df_human_answered

# df_human_answered["Age_3grp"]

In [28]:


p_vals = []
H_vals = []

age_levels = df_human_answered["Age_3grp"].dropna().unique()

for q in likert_cols:
    groups = [
        df_human_answered.loc[
            df_human_answered["Age_3grp"] == g, q
        ].dropna()
        for g in age_levels
    ]
    
    # remove empty groups
    groups = [x for x in groups if len(x) > 0]

    if len(groups) >= 2:
        H, p = kruskal(*groups)
    else:
        H, p = np.nan, np.nan

    H_vals.append(H)
    p_vals.append(p)
    

# there is no sig diff withing people within age grpups who answered the quewstions

#



In [29]:

p_vals

[0.8636561885240785,
 0.09591470007962566,
 0.2066762780619738,
 0.8728431800260286,
 0.7122951368637528,
 0.2724364028680747,
 0.6492796100477998,
 0.1733598421444934,
 0.5400830766978082,
 0.20235750922938398,
 0.4632405646201818,
 0.126187768300768,
 0.4077028837199206,
 0.8651028558673521,
 0.6885060989125467,
 0.9848763742121578,
 0.4605886086870894,
 0.037922824241983726,
 0.1400719596151979,
 0.37927826611506105,
 0.11489300105902311,
 0.01767043449576508,
 0.07187865566496435]

In [35]:
# print(sorted(df["Age band"].dropna().unique()))
# print(sorted(df_human_answered["Age_3grp"].dropna().unique()))

In [36]:
results_df = pd.DataFrame({
    "question": likert_cols,
    "H_stat": H_vals,
    "p_value": p_vals
})

pd.set_option("display.max_colwidth", None)

results_df

results_df_sorted = results_df.sort_values("p_value", ascending=True)
results_df_sorted



,question,H_stat,p_value
21,Having a person handle payment makes me feel more confident and reassured about the transaction,8.071725,0.017670
17,I trust human staff to handle payments more accurately than a digital system,6.544404,0.037923
22,I feel more in control of my payment when interacting with a person rather than a digital system,5.265552,0.071879
1,It feels easier to know what to do when a person guides me upon arrival,4.688592,0.095915
20,It is easier to understand the bill when a person explains what I am paying for,4.327508,0.114893
11,I feel reassured when a person confirms my customized order verbally,4.139969,0.126188
18,It is easier to clarify billing or payment questions with a person,3.931198,0.140072
7,"When a digital system fails, having human staff as a fallback makes me feel supported",3.504772,0.173360
9,It is easier to explain my dietary preferences or special needs to a person,3.195439,0.202358
2,Personal interaction with staff at arrival makes the dining experience feel more genuine,3.153203,0.206676


In [ ]:
# Per-question age counts

# for q in likert_cols:
#     print("\n", q)
#     print(
#         df[df[q].notna()]["Age band"]
#         .value_counts()
#         .sort_index()
#     )


In [37]:
# for _, row in results_df_sorted.iterrows():
#     q = row["question"]
#     H = row["H_stat"]
#     p = row["p_value"]

#     print("\n" + "=" * 120)
#     print("QUESTION:")
#     print(q)
#     print("-" * 120)
#     print(f"H statistic: {H:.4f}")
#     print(f"p-value:     {p:.6f}")
#     print("-" * 120)

#     counts = (
#         df[q]
#         .value_counts(dropna=False)
#         .sort_index()
#         .reset_index()
#     )

#     counts.columns = ["response_code", "count"]
#     counts["response_text"] = counts["response_code"].map(LIKERT_LABELS)

#     # identify max count (ignore NaN response_code)
#     max_count = counts.loc[
#         counts["response_code"].notna(), "count"
#     ].max()

#     # mark the most frequent response(s)
#     counts["most_frequent"] = counts["count"].apply(
#         lambda x: "⭐ most frequent" if x == max_count else ""
#     )

#     print(counts)


In [ ]:
# # total count for human section only
# df_human_answered["Age band"].dropna().value_counts().sum()


In [38]:


# # Only correct non-missing p-values
# mask = results_df["p_value"].notna()

# # BH-adjusted p-values
# results_df.loc[mask, "p_adj_bh"] = multipletests(
#     results_df.loc[mask, "p_value"].values,
#     alpha=0.10,          # choose your FDR level here (e.g., 0.05 or 0.10)
#     method="fdr_bh"
# )[1]
# s
# # Flag significance at 10% FDR (change 0.10 to 0.05 if you want 5%)
# results_df["sig_fdr_10pct"] = results_df["p_adj_bh"] < 0.1

# # Sort by adjusted p-value (recommended for reporting)
# results_df_sorted = results_df.sort_values("p_adj_bh", ascending=True)

# results_df_sorted


In [42]:
from scipy.stats import mannwhitneyu

age_col = "Age_3grp"

pairs = [
    ("18–34", "35–54"),
    ("18–34", "55+"),
    ("35–54", "55+")
]

for q in results_df_sorted["question"]:

    print(f"\n--- {q} ---")

    for g1, g2 in pairs:

        x = df_human_answered.loc[df_human_answered[age_col] == g1, q].dropna()
        y = df_human_answered.loc[df_human_answered[age_col] == g2, q].dropna()

        if len(x) == 0 or len(y) == 0:
            print(f"{g1} vs {g2}: NOT ENOUGH DATA")
            continue

        U, p = mannwhitneyu(x, y, alternative="two-sided")

        print(f"{g1} vs {g2} → p = {p:.4f}")


--- Having a person handle payment makes me feel more confident and reassured about the transaction ---
18–34 vs 35–54 → p = 0.0588
18–34 vs 55+ → p = 0.0122
35–54 vs 55+ → p = 0.2033

--- I trust human staff to handle payments more accurately than a digital system ---
18–34 vs 35–54 → p = 0.1404
18–34 vs 55+ → p = 0.0220
35–54 vs 55+ → p = 0.1529

--- I feel more in control of my payment when interacting with a person rather than a digital system ---
18–34 vs 35–54 → p = 0.3118
18–34 vs 55+ → p = 0.0325
35–54 vs 55+ → p = 0.1189

--- It feels easier to know what to do when a person guides me upon arrival ---
18–34 vs 35–54 → p = 0.0619
18–34 vs 55+ → p = 0.1073
35–54 vs 55+ → p = 0.7547

--- It is easier to understand the bill when a person explains what I am paying for ---
18–34 vs 35–54 → p = 0.2503
18–34 vs 55+ → p = 0.0773
35–54 vs 55+ → p = 0.1540

--- I feel reassured when a person confirms my customized order verbally ---
18–34 vs 35–54 → p = 0.6659
18–34 vs 55+ → p = 0.0488
3

In [48]:
# def print_crosstab_pretty(q, H, p, df_use):
#     ct = pd.crosstab(df_use["Age_3grp"], df_use[q])
#     ct = ct.reindex(columns=[1,2,3,4,5,6], fill_value=0).rename(columns=LIKERT_LABELS)

#     headers = ["Age group"] + list(ct.columns)
#     rows = [[idx] + [int(v) for v in ct.loc[idx].values] for idx in ct.index]

#     # column widths
#     widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]

#     def fmt_row(r):
#         return " | ".join(str(val).ljust(w) for val, w in zip(r, widths))

#     print("\nQUESTION:")
#     print(q)
#     print(f"H = {H:.4f} | p = {p:.6f}\n")
#     print(fmt_row(headers))
#     print("-" * (sum(widths) + 3*(len(widths)-1)))
#     for r in rows:
#         print(fmt_row(r))

# for _, row in results_df_sorted.iterrows():
#     q = row["question"]
#     print_crosstab_pretty(q, row["H_stat"], row["p_value"], df_human_answered)


In [47]:

# import numpy as np
# import pandas as pd
# from itertools import combinations
# from scipy.stats import mannwhitneyu
# from statsmodels.stats.multitest import multipletests

# for q in likert_cols:
#     df_human_answered[q] = pd.to_numeric(df_human_answered[q], errors="coerce")


# age_groups = ["18–34", "35–54", "55+"]
# age_pairs = list(combinations(age_groups, 2))

# pairwise_human = []

# for q in likert_cols:
#     for g1, g2 in age_pairs:
#         x = df_human_answered.loc[df_human_answered["Age_3grp"] == g1, q].dropna().to_numpy(dtype=float)
#         y = df_human_answered.loc[df_human_answered["Age_3grp"] == g2, q].dropna().to_numpy(dtype=float)

#         n1, n2 = len(x), len(y)

#         if n1 < 3 or n2 < 3:
#             U, p = np.nan, np.nan
#         else:
#             U, p = mannwhitneyu(x, y, alternative="two-sided")

#         pairwise_human.append({
#             "question": q,
#             "group_1": g1,
#             "group_2": g2,
#             "n_1": n1,
#             "n_2": n2,
#             "U_stat": U,
#             "p_value": p
#         })

# pairwise_human_df = pd.DataFrame(pairwise_human)
# pairwise_human_df
